# 02 - Construction de la table de référence des dimensions

**Objectif :** transformer les 125 pièces annotées (4 tracés) en table de référence
des dimensions nominales, utilisée plus tard pour comparer un nouveau tracé et
rendre un verdict conforme/non-conforme.

**Rappel important :** les 4 tracés correspondent à 4 modèles de vêtement différents
(codes internes distincts). La référence est donc construite **par modèle** — un
nouveau tracé à tester devra être comparé à la référence du même modèle, identifié
par son code interne.

**Étapes :**
1. Charger les 4 fichiers d'annotations + calculer l'échelle cm/px de chaque tracé
2. Calculer les dimensions physiques (largeur, hauteur, périmètre, aire) de chaque pièce
3. Regrouper par nom de pièce au sein de chaque modèle (moyenne si plusieurs instances)
4. Contrôle qualité : détecter les incohérences entre instances d'une même pièce
5. Sauvegarder la table de référence finale


In [1]:
import sys
sys.path.append('../src')

from pathlib import Path
from database.reference_table import build_full_reference_table, qa_check, save_reference_table

REFERENCE_DIR = Path('../data/raw/reference')
ANNOTATIONS_DIR = Path('../data/processed')

tracees = {
    'Tracee1': (str(REFERENCE_DIR / 'Tracee1.pdf'), str(ANNOTATIONS_DIR / 'Tracee1_annotations.json')),
    'Tracee2': (str(REFERENCE_DIR / 'Tracee2.pdf'), str(ANNOTATIONS_DIR / 'Tracee2_annotations.json')),
    'Tracee3': (str(REFERENCE_DIR / 'Tracee3.pdf'), str(ANNOTATIONS_DIR / 'Tracee3_annotations.json')),
    'Tracee4': (str(REFERENCE_DIR / 'Tracee4.pdf'), str(ANNOTATIONS_DIR / 'Tracee4_annotations.json')),
}

for name, (pdf, ann) in tracees.items():
    assert Path(pdf).exists(), f"PDF manquant: {pdf}"
    assert Path(ann).exists(), f"Annotations manquantes: {ann}"
print("Tous les fichiers sont presents.")

Tous les fichiers sont presents.


## 1-3. Construction de la table de référence

In [2]:
table = build_full_reference_table(tracees)

for name, model_ref in table.items():
    print(f"{name} (code modele: {model_ref.code_modele}) -> {len(model_ref.pieces)} pieces uniques")

Tracee1 (code modele: 157380CD-PDF) -> 15 pieces uniques
Tracee2 (code modele: 02110170811CD-A) -> 14 pieces uniques
Tracee3 (code modele: 01461170745CD-A1) -> 14 pieces uniques
Tracee4 (code modele: 01110170919CD-A) -> 24 pieces uniques


## 4. Contrôle qualité : cohérence entre instances d'une même pièce

In [3]:
all_warnings = []
for name, model_ref in table.items():
    warnings = qa_check(model_ref, tolerance_pct=5.0)
    all_warnings.extend(warnings)

if all_warnings:
    print(f"{len(all_warnings)} avertissement(s) :\n")
    for w in all_warnings:
        print(" -", w)
else:
    print("Aucune incoherence detectee (toutes les pieces repetees ont des dimensions cohérentes entre leurs instances, tolerance 5%).")

1 avertissement(s) :

 - [Tracee2] 'PASS' : hauteur incoherente entre ses 2 instances (moyenne 3.5cm, ecart-type 0.22cm) - a verifier


## Aperçu détaillé d'un modèle (exemple : Tracee1)

In [4]:
import pandas as pd

model = table['Tracee1']
rows = []
for nom, ref in sorted(model.pieces.items()):
    rows.append({
        'piece': nom,
        'instances': ref.n_instances,
        'largeur_cm': ref.largeur_cm,
        'hauteur_cm': ref.hauteur_cm,
        'perimetre_cm': ref.perimetre_cm,
        'aire_cm2': ref.aire_cm2,
    })
df = pd.DataFrame(rows)
df

,piece,instances,largeur_cm,hauteur_cm,perimetre_cm,aire_cm2
0,CHAD-BACK,1,9.33,30.47,71.71,190.63
1,CHAD-COIN PCKT-05,1,13.28,12.37,46.65,148.18
2,CHAD-WB,1,11.91,120.26,264.16,1426.23
3,FACING-03 CHAD-LEFT PATCH,1,16.83,20.08,63.82,261.36
4,FACING-04 CHAD-RIGHT,1,16.88,19.98,63.71,261.85
5,M0YA44-,1,113.32,36.30,280.01,3118.53
6,M0YA44-FLY,1,19.93,6.49,50.88,124.02
7,M0YA44-LBACK10,2,112.96,48.20,291.38,3939.57
8,M0YA44-LFRONT01,1,112.86,36.66,277.20,3088.56
9,M0YA44-UNDERFLY,1,21.55,12.47,67.66,260.02


## 5. Sauvegarde de la table de référence finale

In [5]:
save_reference_table(table, '../data/processed/reference_dimensions.json')

n_total_pieces = sum(len(m.pieces) for m in table.values())
n_total_instances = sum(sum(p.n_instances for p in m.pieces.values()) for m in table.values())
print(f"Table de reference sauvegardee : ../data/processed/reference_dimensions.json")
print(f"{len(table)} modeles, {n_total_pieces} pieces uniques (types), {n_total_instances} instances mesurees au total.")

Table de reference sauvegardee : ../data/processed/reference_dimensions.json
4 modeles, 67 pieces uniques (types), 125 instances mesurees au total.


## Prochaines étapes

1. Utiliser cette table comme référence dans le module `src/quality/` (comparaison,
   verdict conforme/non-conforme, tolérances par type de dimension)
2. Développer le pipeline de mesure automatique (à terme, U-Net) pour extraire les
   dimensions d'un **nouveau** tracé de test de la même façon
3. Identifier automatiquement le modèle d'un nouveau tracé via son code interne pour
   savoir à quelle référence le comparer
